# Jiaozi + Agentic Kaggle Training on Colab GPU

This notebook trains the Jiaozi-generated Plant Pathology 2020 FGVC7 project on Colab.

Flow:

1. Clone `Isso-W/Jiaozi` from the generated integration branch.
2. Configure Colab secrets for Kaggle, Hugging Face, and OpenAI/ChatGPT.
3. Prepare Plant Pathology 2020 data, materialized labels, and stable folds.
4. Inspect the Jiaozi-generated Module 4 candidates.
5. Run a small GPU sweep.
6. Train the selected config longer.
7. Create `submission.csv`, write `submission_receipt.json`, and optionally submit to Kaggle.

The training code is expected under `experiments/plant_pathology_2020_fgvc7_agentic/` in the cloned branch.


## Before Running

In Colab, set these secrets in the left sidebar:

- `KAGGLE_API_TOKEN`, `KAGGLE_JSON`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`
- `HF_TOKEN` for gated Hugging Face checkpoints such as DINOv3
- `OPENAI_API_KEY` or `CHATGPT_API_KEY` if you later want to re-run LLM-assisted generation

Also accept the competition rules once on Kaggle:

`https://www.kaggle.com/c/plant-pathology-2020-fgvc7`


In [ ]:
# Cell 1: clone repo and install dependencies
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path


def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.search(r'\[[^\]]+\]\((https?://[^)\s]+)\)', value)
    if markdown_match:
        return markdown_match.group(1)
    plain_url_match = re.search(r'https?://[^\s)\]]+', value)
    if plain_url_match:
        return plain_url_match.group(0).rstrip(')')
    return value


def remote_ref_exists(repo_url: str, repo_ref: str) -> bool:
    for pattern in (f'refs/heads/{repo_ref}', f'refs/tags/{repo_ref}', repo_ref):
        result = subprocess.run(
            ['git', 'ls-remote', '--exit-code', repo_url, pattern],
            capture_output=True,
            text=True,
        )
        if result.returncode == 0 and result.stdout.strip():
            return True
    return False


def describe_remote_branches(repo_url: str, limit: int = 30) -> str:
    result = subprocess.run(['git', 'ls-remote', '--heads', repo_url], capture_output=True, text=True)
    if result.returncode:
        return result.stderr.strip() or 'Unable to list remote branches.'
    branches = [line.rsplit('refs/heads/', 1)[-1] for line in result.stdout.splitlines() if 'refs/heads/' in line]
    if not branches:
        return 'No remote branches were visible.'
    shown = branches[:limit]
    suffix = f'\n  ... {len(branches) - limit} more' if len(branches) > limit else ''
    return '\n'.join(f'  - {branch}' for branch in shown) + suffix


REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'codex/plant-pathology-producer-consumer-ensemble')
REPO_DIR = Path('/content/Jiaozi')
EXPERIMENT_REL = Path('experiments/plant_pathology_2020_fgvc7_agentic')
EXPERIMENT_DIR = REPO_DIR / EXPERIMENT_REL
MODULE4_DIR = EXPERIMENT_DIR / 'module4_code'

print('REPO_URL =', REPO_URL)
print('REPO_REF =', REPO_REF)
if not remote_ref_exists(REPO_URL, REPO_REF):
    raise RuntimeError(
        f'GitHub cannot see REPO_REF={REPO_REF!r} in {REPO_URL}.\n'
        'Rerun after pushing that branch, or set JIAOZI_REPO_REF to one of these visible branches before Cell 1:\n'
        f'{describe_remote_branches(REPO_URL)}'
    )

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)]
completed = subprocess.run(clone_cmd, capture_output=True, text=True)
print('=== git stdout ===')
print(completed.stdout)
print('=== git stderr ===')
print(completed.stderr)
completed.check_returncode()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if not EXPERIMENT_DIR.exists():
    raise FileNotFoundError(
        f'{EXPERIMENT_DIR} was not found. Push the generated experiment folder to {REPO_REF} first, '
        'or set JIAOZI_REPO_REF to a branch that contains it.'
    )

print('Current commit:')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)

# Avoid reinstalling Colab's CUDA torch unless you explicitly want to.
packages = [
    'chromadb',
    'datasets',
    'huggingface_hub',
    'kaggle>=2.2.2',
    'networkx',
    'openai>=2.0.0',
    'pandas',
    'pillow',
    'scikit-learn',
    'sentence-transformers',
    'timm',
    'transformers',
]
print('\n=== installing dependencies ===')
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages],
    capture_output=True,
    text=True,
)
print(pip_result.stdout)
print(pip_result.stderr)
print('pip return code =', pip_result.returncode)
pip_result.check_returncode()

print('Jiaozi repository:', REPO_DIR)
print('Experiment folder:', EXPERIMENT_DIR)


In [ ]:
# Cell 2: configure Colab secrets
import json
import os
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(names, required: bool = False) -> str:
    if isinstance(names, str):
        names = [names]
    for name in names:
        value = os.getenv(name, '').strip()
        if value:
            return value
    if userdata is not None:
        for name in names:
            try:
                value = (userdata.get(name) or '').strip()
            except Exception:
                value = ''
            if value:
                return value
    if required:
        return getpass(f'{"/".join(names)} (hidden input): ').strip()
    return ''

# OpenAI / ChatGPT API: optional for this generated notebook, useful if you re-run Jiaozi generation later.
openai_api_key = read_secret(['OPENAI_API_KEY', 'CHATGPT_API_KEY'], required=False)
if openai_api_key:
    os.environ['OPENAI_API_KEY'] = openai_api_key
    os.environ.setdefault('JIAOZI_LLM_PROVIDER', 'openai')
    os.environ.setdefault('M1_LLM_PROVIDER', 'openai')
    os.environ.setdefault('M4_LLM_PROVIDER', 'openai')
    print('OpenAI/ChatGPT API key configured.')
else:
    print('No OpenAI/ChatGPT API key found; generated training code can still run.')

# Hugging Face token: recommended for DINOv3 checkpoints.
hf_token = read_secret(['HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN'], required=False)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
    except Exception as exc:
        print('HF login skipped/failed:', exc)
    print('HF token configured.')
else:
    print('No HF token found. DINOv3 may fail if the checkpoint is gated or rate-limited.')

# Kaggle credentials for the kaggle Python package.
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_json = read_secret('KAGGLE_JSON', required=False)
kaggle_api_token = read_secret(['KAGGLE_API_TOKEN', 'KAGGLE_TOKEN'], required=False)
kaggle_username = read_secret('KAGGLE_USERNAME', required=False)
kaggle_key = read_secret(['KAGGLE_KEY', 'KAGGLE_API_KEY'], required=False)


def write_kaggle_json(username: str, key: str) -> None:
    kaggle_json_path = kaggle_dir / 'kaggle.json'
    kaggle_json_path.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
    kaggle_json_path.chmod(0o600)
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key


def write_kaggle_access_token(token: str) -> None:
    access_token_path = kaggle_dir / 'access_token'
    access_token_path.write_text(token.strip(), encoding='utf-8')
    access_token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = token.strip()


def parse_kaggle_json(raw: str, source: str) -> dict:
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f'{source} must contain the downloaded kaggle.json payload. '
            'Use KAGGLE_API_TOKEN for a bare token, or set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets.'
        ) from exc
    if not parsed.get('username') or not parsed.get('key'):
        raise RuntimeError(f'{source} must contain both username and key fields.')
    return parsed


if kaggle_username and kaggle_key:
    write_kaggle_json(kaggle_username, kaggle_key)
    print('Kaggle username/key configured at ~/.kaggle/kaggle.json')
elif kaggle_json:
    parsed = parse_kaggle_json(kaggle_json, 'KAGGLE_JSON')
    write_kaggle_json(parsed['username'], parsed['key'])
    print('Kaggle JSON configured at ~/.kaggle/kaggle.json')
elif kaggle_api_token:
    if kaggle_api_token.lstrip().startswith('{'):
        parsed = parse_kaggle_json(kaggle_api_token, 'KAGGLE_API_TOKEN')
        write_kaggle_json(parsed['username'], parsed['key'])
        print('Kaggle API token JSON configured at ~/.kaggle/kaggle.json')
    else:
        write_kaggle_access_token(kaggle_api_token)
        print('Kaggle API token configured at ~/.kaggle/access_token')
else:
    raise RuntimeError('No Kaggle credential secret found. Add KAGGLE_API_TOKEN, KAGGLE_JSON, or KAGGLE_USERNAME/KAGGLE_KEY in Colab Secrets.')

del openai_api_key, hf_token, kaggle_api_token, kaggle_json, kaggle_username, kaggle_key


In [ ]:
# Cell 3: GPU, Drive, and cache setup
import os
from pathlib import Path

import torch

MOUNT_DRIVE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/Jiaozi')

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Runtime -> Change runtime type -> GPU is recommended before training.')

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        hf_cache = DRIVE_ROOT / 'hf_cache'
        hf_cache.mkdir(parents=True, exist_ok=True)
        os.environ['HF_HOME'] = str(hf_cache)
        os.environ['HF_DATASETS_CACHE'] = str(hf_cache / 'datasets')
        print('HF_HOME =', os.environ['HF_HOME'])
    except Exception as exc:
        print('Drive mount skipped/failed:', exc)
else:
    print('Drive mount disabled.')


In [ ]:
# Cell 4: prepare Kaggle data, labels, folds, and generated configs
import hashlib
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)
DATA_ROOT = Path('/content/kaggle_data')
KAGGLE_COMPETITION = 'plant-pathology-2020-fgvc7'

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_kaggle_secret_token() -> tuple[str, str]:
    if userdata is not None:
        for name in ('KAGGLE_API_TOKEN', 'KAGGLE_TOKEN'):
            try:
                token = (userdata.get(name) or '').strip()
            except Exception:
                token = ''
            if token:
                return token, f'Colab Secret {name}'
    token = os.getenv('KAGGLE_API_TOKEN', '').strip()
    if token:
        return token, 'environment variable KAGGLE_API_TOKEN'
    return '', ''


def ensure_kaggle_access_token() -> dict:
    token, source = read_kaggle_secret_token()
    if not token:
        raise RuntimeError(
            'No Kaggle token found. Add KAGGLE_API_TOKEN in Colab Secrets and enable notebook access.'
        )

    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    access_token_path = kaggle_dir / 'access_token'
    access_token_path.unlink(missing_ok=True)
    access_token_path.write_text(token, encoding='utf-8')
    access_token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = token

    digest = hashlib.sha256(token.encode('utf-8')).hexdigest()[:12]
    print(f'Kaggle API token configured from {source}. length={len(token)} sha256[:12]={digest}')
    return {'token': token, 'source': source, 'digest': digest}


print('\n=== upgrading kaggle package ===')
upgrade_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle>=2.2.2'],
    capture_output=True,
    text=True,
)
print(upgrade_result.stdout)
print(upgrade_result.stderr)
upgrade_result.check_returncode()
kaggle_auth = ensure_kaggle_access_token()

print('\n=== checking kaggle auth ===')
auth_env = os.environ.copy()
auth_env['KAGGLE_API_TOKEN'] = kaggle_auth['token']
auth_check = subprocess.run(
    [sys.executable, '-m', 'kaggle', 'competitions', 'files', '-c', KAGGLE_COMPETITION],
    capture_output=True,
    text=True,
    env=auth_env,
)
print(auth_check.stdout)
print(auth_check.stderr)
if auth_check.returncode:
    raise RuntimeError(
        'Kaggle authentication failed with the token currently visible to this runtime. '
        'If you just regenerated the token, restart the Colab runtime or confirm the printed sha256 prefix changed. '
        'Because the token was shown in a screenshot, regenerate it again after this works.'
    )

prepare_cmd = [
    sys.executable,
    str(EXPERIMENT_DIR / 'colab_prepare_train.py'),
    '--data-root', str(DATA_ROOT),
    '--folds', '5',
    '--fold-index', '0',
    '--image-size', '224',
    '--batch-size', '8',
    '--eval-batch-size', '16',
]
print('Running:', ' '.join(prepare_cmd))
completed = subprocess.run(prepare_cmd, cwd=str(REPO_DIR), capture_output=True, text=True, env=auth_env)
print('=== prepare stdout ===')
print(completed.stdout)
print('=== prepare stderr ===')
print(completed.stderr)
if completed.returncode:
    raise RuntimeError(f'Prepare step failed with exit code {completed.returncode}. See stdout/stderr above.')

print('\nPrepared files:')
prepared_paths = [
    EXPERIMENT_DIR / 'kaggle_run_manifest.json',
    MODULE4_DIR / 'configs.json',
    DATA_ROOT / KAGGLE_COMPETITION / 'train__jiaozi_labels.csv',
    DATA_ROOT / KAGGLE_COMPETITION / 'folds.json',
]
missing_prepared = []
for path in prepared_paths:
    exists = path.exists()
    print(' -', path, 'exists=', exists)
    if not exists:
        missing_prepared.append(path)
if missing_prepared:
    raise FileNotFoundError('Prepare did not create required files: ' + ', '.join(str(p) for p in missing_prepared))


In [ ]:
# Cell 5: inspect generated configs
import json
from pathlib import Path

import pandas as pd

configs = json.loads((MODULE4_DIR / 'configs.json').read_text(encoding='utf-8'))
print('Candidate configs:', len(configs))
rows = []
for index, cfg in enumerate(configs, start=1):
    mc = cfg.get('model_config', cfg)
    rows.append({
        'index': index,
        'rank': cfg.get('rank'),
        'backbone': mc.get('backbone'),
        'checkpoint': mc.get('pretrained_hf_id') or mc.get('checkpoint'),
        'finetune_strategy': mc.get('finetune_strategy'),
        'unfreeze_last_n_blocks': mc.get('unfreeze_last_n_blocks'),
        'loss': mc.get('loss'),
        'metric': mc.get('evaluation_metric'),
        'image_size': mc.get('image_size'),
        'batch_size': mc.get('batch_size'),
        'class_weights': mc.get('use_class_weights'),
        'fold_file': mc.get('fold_file'),
    })

df = pd.DataFrame(rows)
display(df)
print('\nModule 4 summary:')
summary_path = MODULE4_DIR / 'module4_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8')[:6000])
else:
    print('No module4_summary.json found.')


In [ ]:
# Cell 6: optional Hugging Face checkpoint availability check
import json
import os

from huggingface_hub import HfApi

api = HfApi()
configs = json.loads((MODULE4_DIR / 'configs.json').read_text(encoding='utf-8'))
model_ids = []
for cfg in configs:
    mc = cfg.get('model_config', cfg)
    hf_id = mc.get('pretrained_hf_id') or mc.get('checkpoint')
    if hf_id and '/' in str(hf_id) and hf_id not in model_ids:
        model_ids.append(hf_id)

for model_id in model_ids:
    try:
        info = api.model_info(model_id, token=os.getenv('HF_TOKEN') or None)
        print('OK:', model_id, 'private=', getattr(info, 'private', None), 'gated=', getattr(info, 'gated', None))
    except Exception as exc:
        print('FAILED:', model_id, exc)
        print('If this is DINOv3, accept the model terms on Hugging Face and provide HF_TOKEN in Colab Secrets.')


In [ ]:
# Cell 7: small GPU sweep
import copy
import json
import os
import subprocess
import sys
from pathlib import Path

SWEEP_EPOCHS = 3
SWEEP_MAX_CANDIDATES = 4
SWEEP_MAX_TRAIN_SAMPLES = 1200
SWEEP_MAX_EVAL_SAMPLES = 300

HYPERPARAM_SWEEP = [
    {'image_size': 224, 'learning_rate': 1e-4, 'batch_size': 8, 'eval_batch_size': 16},
    {'image_size': 224, 'learning_rate': 5e-5, 'batch_size': 8, 'eval_batch_size': 16},
]


def is_sweep_row(item) -> bool:
    if not isinstance(item, dict):
        return False
    row_keys = {'rank', 'backbone', 'metric_name', 'metric_value', 'selected_best', 'status'}
    return any(key in item for key in row_keys)


def is_sweep_rows(obj) -> bool:
    return isinstance(obj, list) and bool(obj) and all(is_sweep_row(item) for item in obj)


def extract_sweep_rows(text: str):
    decoder = json.JSONDecoder()
    best = None
    for i, char in enumerate(text):
        if char not in '[{':
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            obj = obj.get('rows') or obj.get('results') or [obj]
        if is_sweep_rows(obj) and (best is None or len(obj) > len(best)):
            best = obj
    return best


def ensure_prepared_data() -> None:
    required = [
        MODULE4_DIR / 'configs.json',
        DATA_ROOT / KAGGLE_COMPETITION / 'train__jiaozi_labels.csv',
        DATA_ROOT / KAGGLE_COMPETITION / 'folds.json',
    ]
    missing = [path for path in required if not path.exists()]
    if not missing:
        return

    print('Missing prepared data files:')
    for path in missing:
        print(' -', path)
    print('Re-running prepare step before sweep...')
    prepare_cmd = [
        sys.executable,
        str(EXPERIMENT_DIR / 'colab_prepare_train.py'),
        '--data-root', str(DATA_ROOT),
        '--folds', '5',
        '--fold-index', '0',
        '--image-size', '224',
        '--batch-size', '8',
        '--eval-batch-size', '16',
    ]
    prepare_env = globals().get('auth_env', os.environ.copy())
    prepare_result = subprocess.run(
        prepare_cmd,
        cwd=str(REPO_DIR),
        capture_output=True,
        text=True,
        env=prepare_env,
    )
    print('=== prepare stdout ===')
    print(prepare_result.stdout)
    print('=== prepare stderr ===')
    print(prepare_result.stderr)
    prepare_result.check_returncode()

    still_missing = [path for path in required if not path.exists()]
    if still_missing:
        raise FileNotFoundError('Prepare step finished but files are still missing: ' + ', '.join(str(p) for p in still_missing))


ensure_prepared_data()

base_configs = json.loads((MODULE4_DIR / 'configs.json').read_text(encoding='utf-8'))
sweep_configs = []
for base_index, base_cfg in enumerate(base_configs[:SWEEP_MAX_CANDIDATES], start=1):
    for hp_index, hp in enumerate(HYPERPARAM_SWEEP, start=1):
        cfg = copy.deepcopy(base_cfg)
        mc = cfg.setdefault('model_config', {})
        patch = {
            **hp,
            'offline_smoke': False,
            'max_train_samples': SWEEP_MAX_TRAIN_SAMPLES,
            'max_eval_samples': SWEEP_MAX_EVAL_SAMPLES,
            'validation_fraction': 0.2,
            'checkpoint_dir': str(MODULE4_DIR / 'sweep_checkpoints' / f'candidate_{base_index}_hp_{hp_index}'),
            'resume_checkpoint': '',
        }
        for key, value in patch.items():
            cfg[key] = value
            mc[key] = value
        sweep_configs.append(cfg)

sweep_path = MODULE4_DIR / 'sweep_configs.json'
results_path = MODULE4_DIR / 'sweep_results.json'
sweep_path.write_text(json.dumps(sweep_configs, indent=2, ensure_ascii=False), encoding='utf-8')
if results_path.exists():
    results_path.unlink()
print('Wrote sweep configs:', sweep_path)
print('Total trials:', len(sweep_configs))

cmd = [
    sys.executable,
    '-u',
    'run_experiments.py',
    '--input', str(sweep_path),
    '--epochs', str(SWEEP_EPOCHS),
    '--output', str(results_path),
]
print('Running:', ' '.join(cmd), 'cwd=', MODULE4_DIR)
result = subprocess.run(cmd, cwd=str(MODULE4_DIR), capture_output=True, text=True)
print('=== sweep stdout ===')
print(result.stdout)
print('=== sweep stderr ===')
print(result.stderr)
print('return code =', result.returncode)
result.check_returncode()

if results_path.exists():
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    if not is_sweep_rows(rows):
        raise RuntimeError(f'{results_path} did not contain a list of result rows.')
    print('Wrote:', results_path)
else:
    rows = extract_sweep_rows(result.stdout)
    if rows is None:
        print('Could not parse sweep JSON rows from stdout, but best_config.json may still exist.')
    else:
        results_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
        print('Wrote:', results_path)

best_config_path = MODULE4_DIR / 'best_config.json'
if not best_config_path.exists():
    raise FileNotFoundError('run_experiments.py did not create best_config.json')
print('Best config:', best_config_path)


In [ ]:
# Cell 8: inspect sweep results and best config
import json
from pathlib import Path

import pandas as pd


def normalize_sweep_rows(raw):
    if isinstance(raw, dict):
        raw = raw.get('rows') or raw.get('results') or [raw]
    if not isinstance(raw, list):
        print('sweep_results.json did not contain a list of rows:', type(raw).__name__)
        return []
    row_keys = {'rank', 'backbone', 'metric_name', 'metric_value', 'selected_best', 'status'}
    rows = [item for item in raw if isinstance(item, dict) and any(key in item for key in row_keys)]
    dropped = len(raw) - len(rows)
    if dropped:
        print(f'Ignored {dropped} non-row item(s) in sweep_results.json.')
    return rows


results_path = MODULE4_DIR / 'sweep_results.json'
if results_path.exists():
    rows = normalize_sweep_rows(json.loads(results_path.read_text(encoding='utf-8')))
    if rows:
        df = pd.json_normalize(rows)
        if 'metric_value' in df.columns:
            df['metric_value'] = pd.to_numeric(df['metric_value'], errors='coerce')
        sort_cols = [c for c in ['selected_best', 'metric_value'] if c in df.columns]
        if sort_cols:
            display(df.sort_values(sort_cols, ascending=[False] * len(sort_cols)))
        else:
            display(df)
    else:
        print('No valid sweep result rows found. Re-run Cell 7 with the latest notebook/code.')
else:
    print('No sweep_results.json found.')

best_cfg_path = MODULE4_DIR / 'best_config.json'
if not best_cfg_path.exists():
    raise FileNotFoundError('best_config.json not found. Run Cell 7 first.')
best_cfg = json.loads(best_cfg_path.read_text(encoding='utf-8'))
best_mc = best_cfg.get('model_config', best_cfg)
print(json.dumps({
    'backbone': best_mc.get('backbone'),
    'checkpoint': best_mc.get('pretrained_hf_id') or best_mc.get('checkpoint'),
    'finetune_strategy': best_mc.get('finetune_strategy'),
    'unfreeze_last_n_blocks': best_mc.get('unfreeze_last_n_blocks'),
    'image_size': best_mc.get('image_size'),
    'learning_rate': best_mc.get('learning_rate'),
    'batch_size': best_mc.get('batch_size'),
    'loss': best_mc.get('loss'),
    'metric': best_mc.get('evaluation_metric'),
}, indent=2, ensure_ascii=False))


In [ ]:
# Cell 9: full training with the best config
import json
import os
import subprocess
import sys
from pathlib import Path

FULL_EPOCHS = 15
FULL_USE_ALL_DATA = True

best_cfg = json.loads((MODULE4_DIR / 'best_config.json').read_text(encoding='utf-8'))
mc = best_cfg.setdefault('model_config', {})
mc['offline_smoke'] = False
mc['resume_checkpoint'] = 'auto'
best_cfg['offline_smoke'] = False
best_cfg['resume_checkpoint'] = 'auto'

if FULL_USE_ALL_DATA:
    for key in ('max_train_samples', 'max_eval_samples'):
        mc.pop(key, None)
        best_cfg.pop(key, None)

backbone = mc.get('backbone') or best_cfg.get('backbone') or 'model'
variant = mc.get('strategy_ablation_variant') or best_cfg.get('strategy_ablation_variant') or 'default'
if Path('/content/drive/MyDrive').exists():
    ckpt_dir = DRIVE_ROOT / 'checkpoints' / f'{backbone}_{variant}_plant_pathology_2020'
else:
    ckpt_dir = MODULE4_DIR / 'checkpoints' / f'{backbone}_{variant}_plant_pathology_2020'
ckpt_dir.mkdir(parents=True, exist_ok=True)
mc['checkpoint_dir'] = str(ckpt_dir)
best_cfg['checkpoint_dir'] = str(ckpt_dir)

full_config_path = MODULE4_DIR / 'best_config_full.json'
full_config_path.write_text(json.dumps(best_cfg, indent=2, ensure_ascii=False), encoding='utf-8')
print('Full training config:', full_config_path)
print('Checkpoints:', ckpt_dir)

cmd = [sys.executable, '-u', 'run.py', '--config', str(full_config_path), '--epochs', str(FULL_EPOCHS)]
print('Running:', ' '.join(cmd), 'cwd=', MODULE4_DIR)
subprocess.run(cmd, cwd=str(MODULE4_DIR), text=True).check_returncode()


In [ ]:
# Cell 10: show best checkpoint result
import json
from pathlib import Path

import torch

full_cfg = json.loads((MODULE4_DIR / 'best_config_full.json').read_text(encoding='utf-8'))
flat_cfg = {**full_cfg, **(full_cfg.get('model_config', {}) or {})}
ckpt_dir = Path(flat_cfg.get('checkpoint_dir', MODULE4_DIR / 'checkpoints'))
candidate_paths = [ckpt_dir / 'best_model.pt', MODULE4_DIR / 'checkpoints' / 'best_model.pt']
best_path = next((p for p in candidate_paths if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in candidate_paths]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'))
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))
print('model_load:', json.dumps(ckpt.get('model_load_info'), ensure_ascii=False))

info = ckpt.get('model_load_info') or {}
if info.get('source') == 'tiny_fallback':
    raise RuntimeError('The run used TinyBackbone fallback instead of the requested pretrained checkpoint. Check HF_TOKEN/model access.')


In [ ]:
# Cell 11: prepare submission config for kaggle_submit.py
import json
import shutil
from pathlib import Path

full_config_path = MODULE4_DIR / 'best_config_full.json'
full_cfg = json.loads(full_config_path.read_text(encoding='utf-8'))
flat_cfg = {**full_cfg, **(full_cfg.get('model_config', {}) or {})}

ckpt_dir = Path(flat_cfg.get('checkpoint_dir', MODULE4_DIR / 'checkpoints'))
if not (ckpt_dir / 'best_model.pt').exists():
    raise FileNotFoundError(f'Expected best checkpoint at {ckpt_dir / "best_model.pt"}')

candidate_configs_path = MODULE4_DIR / 'candidate_configs_before_submit.json'
if not candidate_configs_path.exists():
    shutil.copy2(MODULE4_DIR / 'configs.json', candidate_configs_path)

# kaggle_submit.py reads MODULE4_DIR/configs.json, so write the selected final config there.
(MODULE4_DIR / 'configs.json').write_text(json.dumps([full_cfg], indent=2, ensure_ascii=False), encoding='utf-8')
print('Submission config written to:', MODULE4_DIR / 'configs.json')
print('Original candidate configs backed up at:', candidate_configs_path)


In [ ]:
# Cell 12: create submission.csv and optional Kaggle submit
import json
import subprocess
import sys
from pathlib import Path

SUBMIT_TO_KAGGLE = False
LOG_MEMORY_IF_SCORED = True

receipt_path = EXPERIMENT_DIR / 'submission_receipt.json'
submission_cmd = [
    sys.executable,
    'kaggle_submit.py',
    KAGGLE_COMPETITION,
    '--project', str(MODULE4_DIR),
    '--data-root', str(DATA_ROOT),
    '--batch-size', '64',
    '--receipt-out', str(receipt_path),
    '--run-manifest', str(EXPERIMENT_DIR / 'kaggle_run_manifest.json'),
]
if SUBMIT_TO_KAGGLE:
    submission_cmd.append('--submit')
if LOG_MEMORY_IF_SCORED:
    submission_cmd.append('--log-memory')

print('Running:', ' '.join(submission_cmd), 'cwd=', REPO_DIR)
subprocess.run(submission_cmd, cwd=str(REPO_DIR), check=True)

print('\nReceipt:')
print(receipt_path.read_text(encoding='utf-8'))
submission_csv = MODULE4_DIR / 'submission.csv'
print('Submission CSV:', submission_csv, 'exists=', submission_csv.exists())


In [ ]:
# Cell 13: inspect submission shape
from pathlib import Path

import pandas as pd

submission_csv = MODULE4_DIR / 'submission.csv'
if not submission_csv.exists():
    raise FileNotFoundError(submission_csv)
sub = pd.read_csv(submission_csv)
print(sub.shape)
display(sub.head())
required_cols = ['image_id', 'healthy', 'multiple_diseases', 'rust', 'scab']
missing = [col for col in required_cols if col not in sub.columns]
if missing:
    raise ValueError(f'Missing submission columns: {missing}')
prob_cols = required_cols[1:]
assert sub[prob_cols].notna().all().all()
assert ((sub[prob_cols] >= 0) & (sub[prob_cols] <= 1)).all().all()
print('Probability row-sum max error:', float(sub[prob_cols].sum(axis=1).sub(1).abs().max()))


In [ ]:
# Cell 14: optional explicit Kaggle submission poll
POLL_SUBMISSIONS = True

if POLL_SUBMISSIONS:
    import subprocess
    subprocess.run(
        ['kaggle', 'competitions', 'submissions', '-c', KAGGLE_COMPETITION, '-v'],
        check=False,
    )
else:
    print('POLL_SUBMISSIONS=False')


In [ ]:
# Cell 15: package artifacts for download or Drive archival
import shutil
from pathlib import Path

archive_base = Path('/content/plant_pathology_2020_jiaozi_artifacts')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(EXPERIMENT_DIR))
print('Wrote:', archive_path)

if Path('/content/drive/MyDrive').exists():
    target = DRIVE_ROOT / 'plant_pathology_2020_jiaozi_artifacts.zip'
    shutil.copy2(archive_path, target)
    print('Copied to Drive:', target)
